# Cross-Platform Social Media Language Analysis

This project investigates whether users across TikTok, Instagram Reels, and YouTube Shorts exhibit distinct language and emoji usage patterns.

Using comments collected from comparable social media content, I apply natural language processing and machine learning techniques to:

- analyze cross-platform language and emoji usage;
- represent text using n-grams and TF-IDF;
- classify comments by their platform of origin using Multinomial Naive Bayes and Logistic Regression;
- compare model performance and examine the limitations of predicting platform origin from user-generated text.

In [47]:
import re
from collections import Counter

import emoji
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

## 1. Data

The dataset consists of user comments from TikTok, Instagram Reels, and YouTube Shorts. The comments were collected from comparable social media content about carsickness to reduce topic-related differences across platforms.

Each observation contains the text of a comment and its platform of origin.

Because social media comments may contain user-identifying information, the raw dataset is not included in this public repository.

### Loading the Data

The original datasets were collected separately from each platform. Since the exported files used slightly different column names, the relevant text columns are standardized before combining the datasets into a single DataFrame.

In [50]:
# Load platform-specific comment data
youtube = pd.read_csv("data/youtube.txt", sep="\t")
tiktok = pd.read_csv("data/tiktok.txt", sep="\t")
instagram = pd.read_csv("data/instagram.txt", sep="\t")

# Standardize column names
youtube = youtube.rename(columns={"textDisplay": "Comment"})
instagram = instagram.rename(columns={"Comment Text": "Comment"})

# Keep only the columns needed for analysis
youtube = youtube[["Comment", "platform"]]
tiktok = tiktok[["Comment", "platform"]]
instagram = instagram[["Comment", "platform"]]

# Combine all platforms
data = pd.concat([youtube, tiktok, instagram], ignore_index=True)

data.head()

,Comment,platform
0,"If you open a window, the blowing air helps. I...",Youtube Shorts
1,POV: your mom when you drive,Youtube Shorts
2,It can be a vit B deficiency,Youtube Shorts
3,My mom:,Youtube Shorts
4,Me in the back seat istg,Youtube Shorts


### Dataset Overview

Before preprocessing, I examine the size of the combined dataset, the distribution of comments across platforms, and missing values.

In [52]:
print(f"Total comments: {len(data):,}")

print("\nComments by platform:")
print(data["platform"].value_counts())

print("\nMissing values:")
print(data.isnull().sum())

Total comments: 12,391

Comments by platform:
platform
tik tok            5722
Instagram Reels    4947
Youtube Shorts     1722
Name: count, dtype: int64

Missing values:
Comment     15
platform     0
dtype: int64


In [53]:
platform_mapping = {
    "tik tok": "TikTok",
    "Youtube Shorts": "YouTube Shorts",
    "Instagram Reels": "Instagram Reels"
}

data["platform"] = data["platform"].replace(platform_mapping)

data["platform"].value_counts()

platform
TikTok             5722
Instagram Reels    4947
YouTube Shorts     1722
Name: count, dtype: int64

In [54]:
# Remove rows with missing or empty comments
data = data.dropna(subset=["Comment"]).copy()

data["Comment"] = data["Comment"].astype(str).str.strip()
data = data[data["Comment"] != ""].copy()

# Reset the index after cleaning
data = data.reset_index(drop=True)

print(f"Comments after cleaning: {len(data):,}")
print(data["platform"].value_counts())

Comments after cleaning: 12,376
platform
TikTok             5722
Instagram Reels    4932
YouTube Shorts     1722
Name: count, dtype: int64


## 2. Text Preprocessing

Social media comments contain capitalization, emojis, punctuation, and other informal features that require normalization before text classification.

For the classification task, emojis are converted into textual representations rather than removed entirely. This preserves emoji-related information while allowing the symbols to be represented by the text vectorizer.

The preprocessing pipeline:

1. converts text to lowercase;
2. converts emojis to textual descriptions;
3. removes non-word characters;
4. normalizes whitespace.

In [56]:
def preprocess_text(text):
    """Normalize a social media comment for text classification."""
    
    text = str(text).lower()
    text = emoji.demojize(text)
    text = re.sub(r"\W", " ", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

data["processed_comment"] = data["Comment"].apply(preprocess_text)

In [57]:
data[["Comment", "processed_comment"]].head()

,Comment,processed_comment
0,"If you open a window, the blowing air helps. I...",if you open a window the blowing air helps it ...
1,POV: your mom when you drive,pov your mom when you drive
2,It can be a vit B deficiency,it can be a vit b deficiency
3,My mom:,my mom
4,Me in the back seat istg,me in the back seat istg


## 3. Emoji Usage Analysis

Because emojis are an important feature of informal online communication, I also analyze them separately from the classification task.

For this analysis, emojis are extracted from the original comments before text preprocessing. Their frequencies are then compared across platforms.

In [59]:
def extract_emojis(text):
    """Extract emojis from a social media comment."""
    
    return [
        item["emoji"]
        for item in emoji.emoji_list(str(text))
    ]


data["emojis"] = data["Comment"].apply(extract_emojis)

data[["Comment", "emojis"]].head()

,Comment,emojis
0,"If you open a window, the blowing air helps. I...",[]
1,POV: your mom when you drive,[]
2,It can be a vit B deficiency,[]
3,My mom:,[]
4,Me in the back seat istg,[]


In [60]:
emoji_counts = {}

for platform in data["platform"].unique():
    platform_emojis = data.loc[
        data["platform"] == platform, "emojis"
    ].explode().dropna()

    emoji_counts[platform] = Counter(platform_emojis)


for platform, counts in emoji_counts.items():
    print(f"\n{platform}")
    print(counts.most_common(10))


YouTube Shorts
[('😂', 677), ('😭', 147), ('😅', 109), ('❤', 69), ('🤣', 59), ('😢', 45), ('💀', 36), ('🤢', 35), ('🤮', 26), ('😊', 24)]

TikTok
[('😭', 1228), ('😂', 902), ('🤣', 320), ('😅', 168), ('😩', 134), ('💀', 102), ('🤢', 88), ('😫', 54), ('🤮', 35), ('🥺', 34)]

Instagram Reels
[('😂', 2845), ('🤣', 388), ('😭', 286), ('🤢', 201), ('😢', 122), ('😩', 103), ('💀', 68), ('😅', 67), ('❤️', 64), ('👏', 63)]


## 4. Text Classification

To examine whether language patterns differ systematically across platforms, I train machine learning models to predict the platform of origin for each comment.

The processed comments are represented using unigram and bigram features with TF-IDF weighting. I compare two classification algorithms:

- Multinomial Naive Bayes
- Logistic Regression

Model performance is evaluated using accuracy, precision, recall, and F1-score.

In [62]:
X = data["processed_comment"]
y = data["platform"]

### Train, Development, and Test Split

The dataset is divided into training, development, and test sets using an 80/10/10 split. A fixed random seed is used to make the split reproducible.

In [64]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print(f"Training set: {len(X_train)}")
print(f"Development set: {len(X_dev)}")
print(f"Test set: {len(X_test)}")

Training set: 9900
Development set: 1238
Test set: 1238


### Multinomial Naive Bayes

The first model uses a text classification pipeline consisting of:

1. unigram and bigram count features;
2. TF-IDF transformation;
3. Multinomial Naive Bayes classification.

In [66]:
nb_model = Pipeline([
    ("vectorizer", CountVectorizer(ngram_range=(1, 2))),
    ("tfidf", TfidfTransformer()),
    ("classifier", MultinomialNB())
])

nb_model.fit(X_train, y_train)

nb_predictions = nb_model.predict(X_test)

In [67]:
def evaluate_model(y_true, y_pred):
    """Calculate classification performance metrics."""
    
    accuracy = accuracy_score(y_true, y_pred)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

In [68]:
nb_results = evaluate_model(y_test, nb_predictions)

pd.Series(nb_results)

Accuracy     0.615509
Precision    0.670952
Recall       0.615509
F1 Score     0.571300
dtype: float64

### Logistic Regression

The second model uses the same unigram, bigram, and TF-IDF representation, allowing the two classifiers to be compared using the same text features.

In [70]:
lr_model = Pipeline([
    ("vectorizer", CountVectorizer(ngram_range=(1, 2))),
    ("tfidf", TfidfTransformer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

lr_results = evaluate_model(y_test, lr_predictions)

pd.Series(lr_results)

Accuracy     0.638126
Precision    0.659867
Recall       0.638126
F1 Score     0.621629
dtype: float64

In [71]:
results = pd.DataFrame(
    [nb_results, lr_results],
    index=["Multinomial Naive Bayes", "Logistic Regression"]
)

results.round(3)

,Accuracy,Precision,Recall,F1 Score
Multinomial Naive Bayes,0.616,0.671,0.616,0.571
Logistic Regression,0.638,0.660,0.638,0.622


## 5. Results and Interpretation

Logistic Regression achieved the strongest overall performance, with an accuracy of 63.8% and a weighted F1-score of 62.2%. This outperformed Multinomial Naive Bayes, which achieved 61.6% accuracy and a weighted F1-score of 57.1%.

These results suggest that the comments contain some platform-specific linguistic signals that can help distinguish their platform of origin. However, the moderate classification performance also indicates substantial overlap in how users communicate across TikTok, Instagram Reels, and YouTube Shorts.

Several factors may limit model performance:

- **Class imbalance:** The number of comments differs substantially across the three platforms, particularly for YouTube Shorts.
- **Dataset scope:** The comments were collected from a limited set of comparable social media content, so they may not represent broader platform-wide language patterns.
- **Informal language:** Social media comments contain emojis, slang, short expressions, and other highly variable forms of communication.
- **Feature representation:** TF-IDF with unigrams and bigrams captures lexical patterns but does not fully represent context or semantic meaning.

The results should therefore be interpreted as evidence of detectable differences within this dataset rather than as a general-purpose system for identifying the origin of social media comments.

## 6. Conclusion

This project explored whether language and emoji usage differ across TikTok, Instagram Reels, and YouTube Shorts using natural language processing and machine learning.

The analysis combined two perspectives: descriptive analysis of emoji usage and supervised classification of comments by platform. Logistic Regression performed better than Multinomial Naive Bayes, reaching 63.8% accuracy and a weighted F1-score of 62.2%.

Overall, the findings suggest that platform-specific language patterns are detectable, but they are not strong enough in this dataset to reliably determine a comment's platform of origin. A larger and more representative dataset would be needed to evaluate whether these patterns generalize across topics, creators, and online communities.

### Future Work

Possible extensions of this analysis include:

- collecting a larger and more balanced dataset across multiple topics and creators;
- examining which words, n-grams, and emoji features contribute most strongly to platform classification;
- evaluating additional text representations and classification methods;
- analyzing how platform-specific language patterns change across different online communities and content categories.